# Logreg + tf-idf

In [59]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [60]:
train

,ID,url,title,label
0,0,m.kp.md,"Экс-министр экономики Молдовы - главе МИДЭИ, ц...",0
1,1,www.kp.by,Эта песня стала известна многим телезрителям б...,0
2,2,fanserials.tv,Банши 4 сезон 2 серия Бремя красоты смотреть о...,0
3,3,colorbox.spb.ru,Не Беси Меня Картинки,0
4,4,tula-sport.ru,В Новомосковске сыграют следж-хоккеисты алекси...,0
...,...,...,...,...
135304,135304,mail.ru,пора тюльпанов турецкий сериал на русском язык...,0
135305,135305,www.ntv.ru,Остросюжетный сериал «Шеф. Игра на повышение»....,0
135306,135306,topclassiccarsforsale.com,"1941 Plymouth Special Deluxe Hot Rod, Automati...",0
135307,135307,wowcream.ru,Купить It's Skin Сыворотка питательная Power 1...,0


In [61]:
test

,ID,url,title
0,135309,www.kommersant.ru,Шестой кассационный суд в Самаре начнет работу...
1,135310,urexpert.online,"Что такое индексация алиментов, кем и в каких ..."
2,135311,imperimeha.ru,Женщинам | Империя Меха - Part 12
3,135312,national-porn.com,"Небритые, волосатые киски: Порно всех стран и ..."
4,135313,2gis.ru,67
...,...,...,...
165373,300682,etp.armtek.ru,Armtek - запчасти для грузовых и легковых авто...
165374,300683,mail.ru,"Лилия Якупова - Караганда, Карагандинская обла..."
165375,300684,xn----8sbnqchpeeeth.xn--p1ai,Администрация Лесного района Тверской области ...
165376,300685,www-sunhome-ru.cdn.ampproject.org,Сонник Изменение сознания. К чему снится Измен...


In [62]:
train['text'] = (train['url'] + ' ' + train['title']).fillna('').str.lower()
test['text'] = (test['url'] + ' ' + test['title']).fillna('').str.lower()

In [63]:
test

,ID,url,title,text
0,135309,www.kommersant.ru,Шестой кассационный суд в Самаре начнет работу...,www.kommersant.ru шестой кассационный суд в са...
1,135310,urexpert.online,"Что такое индексация алиментов, кем и в каких ...",urexpert.online что такое индексация алиментов...
2,135311,imperimeha.ru,Женщинам | Империя Меха - Part 12,imperimeha.ru женщинам | империя меха - part 12
3,135312,national-porn.com,"Небритые, волосатые киски: Порно всех стран и ...","national-porn.com небритые, волосатые киски: п..."
4,135313,2gis.ru,67,2gis.ru 67
...,...,...,...,...
165373,300682,etp.armtek.ru,Armtek - запчасти для грузовых и легковых авто...,etp.armtek.ru armtek - запчасти для грузовых и...
165374,300683,mail.ru,"Лилия Якупова - Караганда, Карагандинская обла...","mail.ru лилия якупова - караганда, карагандинс..."
165375,300684,xn----8sbnqchpeeeth.xn--p1ai,Администрация Лесного района Тверской области ...,xn----8sbnqchpeeeth.xn--p1ai администрация лес...
165376,300685,www-sunhome-ru.cdn.ampproject.org,Сонник Изменение сознания. К чему снится Измен...,www-sunhome-ru.cdn.ampproject.org сонник измен...


In [64]:
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.linear_model import LogisticRegression

In [65]:
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=20000)
X_train = vectorizer.fit_transform(train['text'])
X_test = vectorizer.transform(test['text'])

y_train = train['label']


In [66]:
X_train.shape, X_test.shape

((135309, 20000), (165378, 20000))

In [67]:
clf = LogisticRegression(penalty='l2')
clf.fit(X_train, y_train)

LogisticRegression()

In [68]:
test_preds = clf.predict(X_test)

In [69]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score, make_scorer

f1 = cross_val_score(clf, X_train, y_train, cv=5, scoring=make_scorer(f1_score)).mean()
print("CV F1 Score:", f1)

C:\Users\User\anaconda3\envs\MLenv\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,
C:\Users\User\anaconda3\envs\MLenv\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mod

CV F1 Score: 0.9513631378386043


In [18]:
submission = pd.DataFrame({
    'ID': test['ID'],
    'label': test_preds
})

submission.to_csv('submission.csv', index=False)

# HashingVectorizer + logreg

In [70]:
vectorizer = HashingVectorizer(n_features=2**14, alternate_sign=False, ngram_range=(1,2), norm='l2')
X_train = vectorizer.fit_transform(train['text'])
X_test = vectorizer.transform(test['text'])

y_train = train['label']

In [71]:
clf = LogisticRegression(penalty='l2', C=1.0, class_weight='balanced', max_iter=1000)
clf.fit(X_train, y_train)

test_preds = clf.predict(X_test)

In [72]:
f1 = cross_val_score(clf, X_train, y_train, cv=5, scoring=make_scorer(f1_score)).mean()
print("CV F1 Score:", f1)

CV F1 Score: 0.9365190338353784


In [34]:
# submission2 = pd.DataFrame({
#     'ID': test['ID'],
#     'label': test_preds
# })

# submission.to_csv('submission2.csv', index=False)

# FastText

In [48]:
import re
import fasttext
import csv

In [42]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [45]:
train['text'] = (train['url'].fillna('') + ' ' + train['title'].fillna('')).str.lower()
test['text'] = (test['url'].fillna('') + ' ' + test['title'].fillna('')).str.lower()

In [46]:
train['label_ft'] = train['label'].apply(lambda x: '__label__' + str(x))

In [49]:
train[['label_ft', 'text']].to_csv(
    'train_fasttext.txt',
    sep=' ',
    header=False,
    index=False,
    quoting=csv.QUOTE_NONE,
    escapechar='\\'
)

In [55]:
model = fasttext.train_supervised(
    input='train_fasttext.txt',
    minn=3,
    maxn=5,
    wordNgrams=2,
    dim=25,
    bucket=200000,
    minCount=5,
)

In [56]:
model.quantize(input='train_fasttext.txt', retrain=True)
model.save_model("fasttext_model_compressed.ftz")

In [57]:
test_preds = [model.predict(t)[0][0].replace('__label__', '') for t in test['text']]
test_preds = list(map(int, test_preds))

In [58]:
# submission3 = pd.DataFrame({
#     'ID': test['ID'],
#     'label': test_preds
# })
# submission.to_csv('submission3.csv', index=False)